# 05 — Feature Engineering & Data Split

## Goal

This notebook exists to **test two things**:

1. **The created features** — that every column produced by the three feature modules is what it claims to be. Correct grain, correct derivation from the source columns, no missing values, no impossible values, and no column that silently encodes something other than its name.
2. **The train / test split** — a time-ordered 80 / 20 split on each customer's first purchase date, followed by PCA fitted on the training customers only. That the split is reproducible, partitions the customers cleanly, puts every training customer before every test customer, and that the PCA never saw the test set.

Neither is exploratory. `04_eda_first_txn.ipynb` asked *what does the data say*; this notebook asks *is what we built correct*. Every cell here should be a check with a pass/fail reading, not a chart to interpret.

## The modules under test

Each one reads the same cleaned line items and produces a table at its own grain. The notebook calls their functions directly rather than reading their CSVs, so what is tested is the code that generates the features, not an artefact that may have gone stale on disk.

| Module | Produces | Grain | Rows |
|---|---|---|---|
| `src/build_features.py` | `customer_features.csv` | `customer_id` | 5,044 |
| `src/build_item_history.py` | `item_history.csv` | `(stock_code, date)` | 100,158 |
| `src/description_embeddings.py` | `description_embeddings.npz` | `stock_code` | 4,171 |

Only the first is a feature table. The other two are **lookups**: they sit at a grain other than the unit of prediction, so neither is a feature of a customer until it is aggregated to one — and how to aggregate it is a modelling decision left to the two pipelines. This notebook joins them onto the line items and stops there.

Each module ships a matching `check_*` function that asserts its way through the properties its table must satisfy. Those run here too, so a cell that executes without raising is itself the first pass/fail reading.

**Source:** `data/processed/first_transaction_churn_clean.csv` — the cleaned line-item table, one row per line item of each customer's first invoice, written by `src/correct_data_issues.py`. Everything below is built from it.

## What is being tested

### The features

`build_features.py` collapses 125,042 corrected line items into 5,044 customer rows across 16 columns. The item-history and embedding lookups are checked further down, where each is built. The checks below verify each group against the line-item source it came from, rather than trusting the aggregation:

| Group | Columns | What has to hold |
|---|---|---|
| Keys & label | `customer_id`, `churn` | one row per customer; label binary and unchanged from the source |
| Calendar parts | `year`, `month`, `day_of_month`, `weekday`, `hour`, `time_of_day` | each recomputable from `first_date`; buckets total and within domain |
| Category | `country`, `country_raw` | every named country above the frequency floor; pooled rows all `Other`; the raw column regroups back to the derived one |
| Roll-ups | `n_lines`, `n_distinct_products`, `total_quantity`, `total_spend`, `avg_unit_price` | each equal to the aggregation recomputed directly from the line items |

### One-hot encoding

`build_modelling_dataset.py` one-hot encodes `weekday` (7 values), `time_of_day` (3) and `country` (14) **before** the split, dropping the most frequent category of each as the baseline (`weekday` 3, `time_of_day` `afternoon`, `country` `United Kingdom`), and drops `country_raw`. That leaves 12 plain customer columns plus 21 dummies. The category lists, and the country frequency floor behind them, are computed over all customers, so they are informed by rows that later become test data; they never look at `churn`.

### The split

`src/split_data.py` orders customers by `first_date`, puts the earliest 80% in train and the rest in test, then fits PCA on the training customers' mean embeddings and applies it unchanged to the test customers. The split has to satisfy these properties:

- **Complete and disjoint** — every customer lands in exactly one of train / test; no customer appears twice.
- **Split on the customer** — the modelling grain is the customer, so the split unit is `customer_id`.
- **Ordered in time** — the latest `first_date` in train is strictly earlier than the earliest in test, and customers sharing the cutoff timestamp stay on the same side.
- **Reproducible** — a second run gives identical train, test and PCA.
- **PCA fitted on train only** — the components and centre come from the training rows alone, and the test rows are projected with them.
- **Nothing else altered** — apart from the embedding columns and the removed `customer_id`, every value equals the modelling table's value for the same customer, row for row; train and test have identical columns and dtypes, and `churn` (the label) and `first_date` (used only to order rows for time-ordered CV, never as a feature) are present in both.
- **One-hot columns valid** — every dummy is 0/1, and the written files match what the code produces.
- **Base rates reported** — `04` §6 shows churn moving between 31.2% and 72.4% month to month, so a time-ordered split will not have equal churn rates in the two parts. That is reported, not asserted: there is no agreed threshold for it.

### Not handled by the split

- The opening cohort (the first 90 days, 1,581 customers churning at 36.1% against 50.0% afterwards) is kept, and it falls entirely in train.
- No 90-day gap separates train from test, so the labels of the last training customers are resolved using weeks that overlap the start of the test period.
- There is no validation set; tuning uses cross-validation inside train.


In [ ]:
# This notebook imports from src/, which is edited while the kernel is alive.
# Without autoreload, `from build_features import ...` is served from the module
# Python cached on first import, so a function or constant added to a .py file
# after the kernel started raises ImportError until the kernel is restarted.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

RANDOM_SEED = 42

PROCESSED = Path('..') / 'data' / 'processed'
SRC = Path.cwd().parent / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# The starting point for everything below: one row per line item of each
# customer's first invoice, after the corrections in correct_data_issues.py.
lines = pd.read_csv(PROCESSED / 'first_transaction_churn_clean.csv',
                    parse_dates=['invoice_date'])

print(f'Cleaned first transactions : {lines.shape[0]:,} rows x {lines.shape[1]} columns')
print(f'Customers                  : {lines["customer_id"].nunique():,}')
print(f'Invoices                   : {lines["invoice"].nunique():,}')
print(f'Date range                 : {lines["invoice_date"].min():%Y-%m-%d} -> '
      f'{lines["invoice_date"].max():%Y-%m-%d}')
print(f'Churn rate                 : {lines.groupby("customer_id")["churn"].first().mean():.2%}')
print()
print(lines.dtypes.to_string())

lines.head(10)

## Building the variables

Two modules turn the table above into modelling inputs, split by grain. The cells below call them directly rather than reading their output:

| Table | Grain | Function |
|---|---|---|
| Customer modelling table | one row per `customer_id` | `build_features(lines)` from `src/build_features.py` |
| Item history lookup | one row per `(stock_code, date)` | `build_item_history(lines)` from `src/build_item_history.py` |

Each has a matching `check_*` function in the same module that asserts its way through the properties the table has to satisfy; both are run here, so a cell that executes without raising is itself the first pass/fail reading.

The item table is a **lookup**, not a feature table — which is why it lives in its own module and is deliberately not joined to the customer grain. How to summarise a basket's worth of product history to one row per customer is a modelling decision. Joining it onto the line items is therefore the next cell's job, and the aggregation to customer grain is left open.

### What gets added to the line items, and what does not

The dividing line is whether a value **varies within a basket**. Only what varies is computed here; everything constant is computed once per customer by `build_features.py` and joined after the collapse.

- **Joined on `(stock_code, date)`** — the six `prior_*` history columns, which differ product by product.
- **Derived per row** — the four line-vs-history ratios, which differ line by line.
- **Not here** — `year`, `month`, `weekday`, `hour`, `time_of_day`, `country`. These are identical on every line of a customer's single transaction, so deriving them at line grain would write the same value 18 times over (the median basket) and discard 17 of them. `build_features.py` derives them *after* grouping to one row per customer, which is also what keeps the country frequency floor counting customers rather than basket lines.
- **Not here either** — the customer-level roll-ups (`n_lines`, `total_quantity`, `total_spend`, `avg_unit_price`, …). Those are the *result* of aggregating these line items; broadcasting them back onto the rows they came from is how double counting starts.

`src/build_modelling_dataset.py` performs the collapse and the join.

In [ ]:
from build_features import add_date_parts, build_features, check_modelling_table
from build_item_history import build_item_history, check_item_history

# Customer grain: one row per customer, the shared starting point for both pipelines.
features = build_features(lines)
check_modelling_table(features)

# (stock_code, date) grain: how each product had traded strictly before that date.
items = build_item_history(lines)
check_item_history(items, lines)

print(f'features : {features.shape[0]:,} rows x {features.shape[1]} columns  '
      f'(checks passed)')
print(f'items    : {items.shape[0]:,} rows x {items.shape[1]} columns  '
      f'(checks passed)')

# Both were built from `lines` here; the module also writes them to disk. Read the
# item lookup back to confirm the file and the function agree, then preview it.
items_file = pd.read_csv(PROCESSED / 'item_history.csv',
                         parse_dates=['date'])
assert items_file.shape == items.shape, 'item_history.csv is a different shape'
pd.testing.assert_frame_equal(items_file, items.reset_index(drop=True))
print('\nitem_history.csv matches build_item_history(lines)')

print(f'\nDistinct stock codes     : {items["stock_code"].nunique():,}')
print(f'Rows with no history yet : {(items["prior_transactions"] == 0).sum():,} '
      f'({(items["prior_transactions"] == 0).mean()*100:.1f}%) — every product\'s '
      f'first appearance')

items_file.head(10)

In [ ]:
items_file.head()

## Item history applied to the first transactions

Applying `src/build_features.py` to the cleaned first-transaction data, so every line item carries the trading history its product had **before** that transaction's date.

### How the metrics move over time

The module produces one row per `(stock_code, date)`, each computed from data strictly earlier than that date. A product bought on 3 March and again on 6 March therefore gets two different rows — the first sees everything up to 2 March, the second everything up to 5 March. The history grows as the product accumulates trade:

| `stock_code` | `date` | `prior_transactions` | `prior_units` |
|---|---|---|---|
| 85123A | 2009-12-01 | 0 | 0 |
| 85123A | 2009-12-03 | 31 | 1,003 |
| 85123A | 2009-12-06 | 61 | 1,415 |

Same-day transactions share a cutoff: because history is aggregated to whole days before any accumulation, all 18 customers who bought `85123A` on 2009-12-08 read the same 79 prior transactions, and none of them sees the other 17.

### The six history metrics

**Volume — how much the product had sold**

- **`prior_transactions`** — transactions containing this product before today.
- **`prior_units`** — units of it sold before today.
- **`prior_median_daily_transactions`** — on a typical earlier trading day, how many transactions included it.
- **`prior_median_daily_units`** — on a typical earlier trading day, how many units moved.

**Price level — what the product normally cost**

- **`prior_avg_price`** — mean, across earlier trading days, of that day's average unit price.
- **`prior_median_daily_price`** — median of the same daily prices, so a one-off promotion does not drag the level.

Both price columns are built from **daily** averages rather than from raw line items, matching how the unit metrics work. A day on which forty customers bought counts once, exactly as a one-customer day does — otherwise busy days would define "the usual price". This matters because `04` §8a found 1,681 codes (40.3%) selling at more than one price.

### Comparing the current line against that history

`add_history_ratios` adds four columns putting the line's own values next to what the product normally does:

- **`qty_vs_median_daily_units`** = `quantity / prior_median_daily_units`. Above 1 means this single line moved more than the product's whole typical day.
- **`qty_share_of_prior_units`** = `quantity / prior_units`. What fraction of everything ever sold of this product is being bought right now.
- **`price_vs_prior_avg_price`** = `price / prior_avg_price`. Above 1 means this customer paid a premium over the product's usual level; below 1 means they caught it discounted.
- **`price_vs_prior_median_daily_price`** = `price / prior_median_daily_price`. The same comparison against the median daily price instead of the mean.

The two price ratios differ only in how the historical level is summarised, and that difference is the point: a one-off promotion or a single odd trading day drags the mean but not the median. They agree for a stably-priced product and diverge for one whose history holds outlier days, so the gap between them is itself a signal about how erratically a product has been priced.

All four are `NaN` on a product's first-ever appearance, where there is no history to divide by. That is roughly 4% of rows and is a genuine undefined, not a missing value to impute.

In [ ]:
# Item history onto every line item. (stock_code, date) is the lookup's key, so a
# many-to-one merge is the whole join: several line items may share a product-day.
txn = lines.assign(date=lines['invoice_date'].dt.normalize())
txn = txn.merge(items, on=['stock_code', 'date'], how='left', validate='many_to_one')

assert len(txn) == len(lines), 'the merge changed the row count'
assert txn['prior_transactions'].notna().all(), 'a line item found no history row'

# Nothing constant within a customer is derived here — no calendar parts, no
# country grouping. Those live in `features`, one row per customer already, and
# are joined after the collapse. Deriving them on 125,042 line items would repeat
# each value once per basket line only to discard all but one of them.
assert (txn.groupby('customer_id')['prior_transactions'].size()
        == features.set_index('customer_id')['n_lines']).all(), \
    'line counts no longer agree with n_lines'

added = [c for c in txn.columns if c not in lines.columns]
print(f'Enriched line items : {txn.shape[0]:,} rows x {txn.shape[1]} columns')
print(f'Added {len(added)} columns: {", ".join(added)}')
print()
print('Every added column varies within a basket, so every one of them has to be '
      'aggregated\nto reach the customer grain. The constant ones come from '
      '`features` at the join.')

txn.head(5)

In [ ]:
from build_item_history import RATIO_COLUMNS, add_history_ratios

# Line-vs-history ratios. Denominators are guarded inside the function: prior_units
# is 0 on a product's first appearance and the medians and means are NaN there, so
# a debut row comes out NaN rather than infinite.
txn = add_history_ratios(txn)

print(f'Line items with history joined : {len(txn):,}')
print(f'Ratio columns added            : {", ".join(RATIO_COLUMNS)}')
print(f'Undefined on a debut row       : '
      f'{txn["qty_share_of_prior_units"].isna().sum():,} '
      f'({txn["qty_share_of_prior_units"].isna().mean()*100:.1f}%)')

# The two price ratios summarise the same history differently, so where they
# disagree the product's price history holds outlier days.
both = txn[RATIO_COLUMNS[2:]].dropna()
gap = (both['price_vs_prior_avg_price'] - both['price_vs_prior_median_daily_price']).abs()
print(f'\nMean vs median price ratio     : identical to 3dp on '
      f'{(gap < 5e-4).mean()*100:.1f}% of rows, '
      f'largest gap {gap.max():.2f}')
print(both.describe(percentiles=[0.5, 0.99]).round(3).to_string())

In [ ]:
ITEM_HISTORY_COLUMNS = [
    'date',
    'customer_id',
    'stock_code',
    'quantity',
    'price',
    'prior_transactions',
    'prior_units',
    'prior_median_daily_transactions',
    'prior_median_daily_units',
    'prior_avg_price',
    'prior_median_daily_price',
    *RATIO_COLUMNS,
]

item_history = txn[ITEM_HISTORY_COLUMNS].sort_values(['date', 'customer_id', 'stock_code'])
print(f'{len(item_history):,} rows x {item_history.shape[1]} columns')

# The opening day is every product's first appearance, so it is all NaN by
# construction and makes a poor preview. Show a mid-period transaction instead.
mid = item_history[item_history['prior_transactions'] > 0]
example_customer = mid.iloc[len(mid) // 2]['customer_id']

print(f'\nOne complete transaction — customer {example_customer}:')
item_history[item_history['customer_id'] == example_customer].head(5)

In [ ]:

# mid['customer_id'].unique()
mid[mid['customer_id'] == 12437]
# mid

## Description embeddings

`src/description_embeddings.py` encodes each product's `description` with SBERT (`all-MiniLM-L6-v2`, 384 dimensions), so text that carries real meaning — `RED HANGING HEART T-LIGHT HOLDER` — stops being an opaque string.

### Grain, again

Descriptions belong to the **product**, not the line item. `correct_data_issues.normalize_descriptions` has already collapsed every `stock_code` to one modal description, so the 125,042 line items hold only 4,171 distinct `(stock_code, description)` pairs. Embedding the line items would encode the same strings thirty times over; the module embeds the catalogue once and leaves the join to the caller, exactly as the item history does.

So this is a third table, at a third grain:

| Table | Grain | Rows |
|---|---|---|
| `features` | `customer_id` | 5,044 |
| `items` | `(stock_code, date)` | 100,158 |
| `products` / `product_embeddings` | `stock_code` | 4,171 |

Turning a 384-dimensional product vector into a feature of a *customer* means choosing how to pool a basket's worth of vectors — mean, max, tf-idf weighting, cluster assignment, distance to the customer's own centroid. That is a modelling decision, so it is left to the two pipelines rather than done here.

### Why the dot product is the similarity

`embed_descriptions` returns L2-normalised rows, so `a @ b` is exactly the cosine similarity of `a` and `b`, bounded in [-1, 1]. On unnormalised vectors a plain dot product would also reward magnitude, which for sentence embeddings tracks description length more than meaning. Normalising once up front means a whole similarity matrix is one matrix multiply.

In [ ]:
from description_embeddings import (MATMUL_ERRSTATE, build_description_embeddings,
                                    check_embeddings, load_model, most_similar)

# One model instance, reused by the next cell too.
sbert = load_model()

# Product grain: 125,042 line items -> one embedding per distinct stock_code.
products, product_embeddings = build_description_embeddings(lines, model=sbert)
check_embeddings(products, product_embeddings)

print(f'Products embedded  : {len(products):,} (from {len(lines):,} line items)')
print(f'Embedding matrix   : {product_embeddings.shape} {product_embeddings.dtype}  '
      f'({product_embeddings.nbytes / 1e6:.1f} MB in memory)')
print(f'Row norms          : all 1.0 '
      f'(max deviation {abs(np.linalg.norm(product_embeddings, axis=1) - 1).max():.2e})')

# Distinct codes can share a description — colour and size variants whose
# distinguishing detail never reached the text. They embed identically, so a
# similarity of exactly 1.0 between two different codes is expected below.
shared = products.groupby('description')['stock_code'].size()
shared = shared[shared > 1]
print(f'Shared descriptions: {len(shared)} descriptions cover {int(shared.sum())} products, '
      f'e.g. {shared.idxmax()!r} x{shared.max()}')

# products.iloc[i] describes product_embeddings[i] — check_embeddings asserts it.
preview = products.head(10).copy()
for d in range(4):
    preview[f'dim_{d}'] = product_embeddings[:10, d]
preview

In [ ]:
# A sample description, taken from the catalogue so its own row appears at
# similarity 1.0 — the end-to-end proof that the matrix rows are still aligned
# with `products`.
QUERY_STOCK_CODE = '85123A'
TOP_N = 10

position = products.index[products['stock_code'] == QUERY_STOCK_CODE][0]
query = products.loc[position, 'description']
query_vec = product_embeddings[position]

# The dot product against every product. Both sides are unit length, so this
# *is* the cosine similarity — no scaling, no normalising, no special function.
# errstate silences spurious Accelerate BLAS flags; see MATMUL_ERRSTATE.
with np.errstate(**MATMUL_ERRSTATE):
    similarity = product_embeddings @ query_vec

order = np.argsort(-similarity)[:TOP_N]
neighbours = (products.iloc[order]
              .assign(similarity=similarity[order])
              .reset_index(drop=True))

print(f'Query: {QUERY_STOCK_CODE}  {query!r}')
print(f'Similarity range over all {len(products):,} products: '
      f'{similarity.min():.3f} to {similarity.max():.3f}, mean {similarity.mean():.3f}')

# The module helper must agree. Compared as sets and sorted values rather than
# row-for-row: identical descriptions give identical similarities, and the two
# code paths (argsort here, argpartition there) may break those ties differently.
helper = most_similar(query, products, product_embeddings, model=sbert, top_n=TOP_N)
assert set(helper['stock_code']) == set(neighbours['stock_code']), 'helper disagrees'
assert np.allclose(sorted(helper['similarity']), sorted(neighbours['similarity'])), \
    'helper similarities differ'
print('most_similar() returns the same neighbours as the dot product above')

# 4 decimals here rather than the notebook-wide 2, so the exact ties are visible.
FMT = '{:.4f}'.format
ties = neighbours['similarity'].duplicated(keep=False)
if ties.any():
    print(f'\n{ties.sum()} of the top {TOP_N} tie exactly — products sharing one description:')
    print(neighbours[ties].to_string(index=False, float_format=FMT))

with pd.option_context('display.float_format', FMT):
    display(neighbours)

### Attaching the embeddings to the line items

The whole 384-dimensional vector goes into **one** column rather than 384 float columns, so the frame stays readable and the vector stays a single object to aggregate.

Every line item of a given product points at the *same* array: the catalogue has 4,171 products behind 125,042 line items, so a per-row copy would inflate 6.4 MB into 192 MB without adding any information. Those shared arrays are marked read-only, since writing through one row would otherwise silently rewrite every other row of the same product.

To reach the customer grain later, stack and average:

```python
customer_vectors = (txn.groupby('customer_id')[EMBEDDING_COLUMN]
                       .apply(lambda s: stack_embeddings(s).mean(axis=0)))
```

One thing to decide when you do: the stored vectors are unit length, but **a mean of unit vectors is not**. Across these customers the mean vector's norm runs from 0.42 to 1.00, median 0.54 — it reaches 1.0 only for a basket whose products all share one description, and falls as the basket gets more varied. That norm is a real signal about basket coherence, not an artefact, so re-normalise only if you want direction without it.

In [ ]:
from description_embeddings import (EMBEDDING_COLUMN, EMBEDDING_DIM,
                                    attach_embeddings, stack_embeddings)

txn = attach_embeddings(txn, products, product_embeddings)
vectors = txn[EMBEDDING_COLUMN]

print(f'Enriched line items : {txn.shape[0]:,} rows x {txn.shape[1]} columns')
print(f'New column          : {EMBEDDING_COLUMN} — one '
      f'{vectors.iloc[0].shape[0]}-dim {vectors.iloc[0].dtype} vector per row')
print(f'Distinct arrays     : {len({id(v) for v in vectors}):,} for {len(txn):,} rows '
      f'(one per product, shared and read-only)')
print(f'Column memory       : {vectors.memory_usage(deep=True) / 1e6:.1f} MB, against '
      f'{len(txn) * EMBEDDING_DIM * 4 / 1e6:.0f} MB if each row held its own copy')

# Every row must carry its own product's catalogue vector, not a neighbour's.
positions = products.set_index('stock_code').index.get_indexer(txn['stock_code'])
assert np.array_equal(stack_embeddings(vectors), product_embeddings[positions]), \
    'a line item is holding the wrong product vector'
print('Every row matches its stock_code\'s catalogue vector')

# --------------------------------------------------------------------------
# The finished line-item table: item history and embeddings side by side
# --------------------------------------------------------------------------
# A subset of the 28 columns — enough to see that each line carries both what
# its product had done before that date and what its description means.
SHOWCASE_COLUMNS = [
    'customer_id', 'stock_code', 'description', 'date', 'quantity', 'price',
    'prior_transactions', 'prior_units', 'prior_median_daily_units',
    'prior_avg_price', 'price_vs_prior_avg_price',
    'price_vs_prior_median_daily_price', EMBEDDING_COLUMN,
]

# One complete transaction, from a customer whose products all have history —
# a product's debut day is NaN by construction and makes a poor preview.
with_history = txn[txn['prior_transactions'] > 0]
example_customer = with_history.iloc[len(with_history) // 2]['customer_id']
showcase = txn[txn['customer_id'] == example_customer][SHOWCASE_COLUMNS].copy()

# Abbreviate the vectors: pandas renders an object cell with str(), which would
# otherwise print all 384 values or clip them mid-number.
showcase[EMBEDDING_COLUMN] = [np.array2string(v, precision=2, threshold=6,
                                              edgeitems=2, suppress_small=True)
                              for v in showcase[EMBEDDING_COLUMN]]

print(f'\nCustomer {example_customer} — {len(showcase)} line items, '
      f'{len(SHOWCASE_COLUMNS)} of {txn.shape[1]} columns shown')
print('  item history  -> prior_* and price_vs_prior_avg_price, from build_item_history.py')
print(f'  embeddings    -> {EMBEDDING_COLUMN}, from description_embeddings.py')

with pd.option_context('display.width', 250, 'display.max_colwidth', 40):
    display(showcase.head(12))

In [ ]:
txn[txn['invoice'] == 489669]

## Assembling the modelling dataset

`src/build_modelling_dataset.py` performs the collapse: line items in, one row per customer out. It composes the three feature modules rather than deriving anything new, in four stages.

| Stage | Function | Result |
|---|---|---|
| 1 | `build_line_metrics(lines)` | line grain, carrying the 6 `prior_*` and the 4 ratios |
| 2 | `aggregate_line_metrics(txn)` | 5,044 x 50 — each metric by min/max/mean/median/sum |
| 3 | `aggregate_embeddings(txn, ...)` | 5,044 x 384 — the basket's mean SBERT vector |
| 4 | one-hot encode and join `customer_features` | 5,044 x 467 |

The cells below run each stage separately and check it against a hand recomputation, then confirm the module's own `build_modelling_dataset` reproduces the same table end to end. Each stage is checked for the thing that could actually go wrong with it:

- **Stage 1** — that only genuinely line-varying columns are carried forward. A column that is constant within a basket has no business being aggregated.
- **Stage 2** — that the five aggregations are what they claim, including the `NaN` handling for a customer whose whole basket lacks history.
- **Stage 3** — that the mean is over the customer's *line items*, so a product bought twice counts twice.
- **Stage 4** — that the one-hot dummies decode back to the original columns, and that the remaining customer-level columns arrive **unchanged**, not silently reordered or recomputed.

In [ ]:
from build_modelling_dataset import (AGGREGATE_COLUMNS, AGGREGATIONS, EMBEDDING_COLUMNS,
                                     LINE_METRIC_COLUMNS, aggregate_embeddings,
                                     aggregate_line_metrics, build_line_metrics,
                                     DROPPED_COLUMNS, ONE_HOT_BASELINES,
                                     build_modelling_dataset, check_modelling_dataset,
                                     dummy_column_name, one_hot_encode)

# ---- Stage 1: the line-varying metrics ------------------------------------
line_metrics = build_line_metrics(lines)

# The module must reproduce exactly what this notebook built by hand in the
# item-history cells above — same rows, same values, for every metric column.
carried = ['date'] + LINE_METRIC_COLUMNS
pd.testing.assert_frame_equal(line_metrics[carried], txn[carried])
print(f'build_line_metrics reproduces the notebook\'s txn on all '
      f'{len(carried)} carried columns')

# Every aggregated column must actually vary within a basket, or aggregating it
# is busywork hiding a constant.
varies = {c: txn.groupby('customer_id')[c].nunique(dropna=False).max() > 1
          for c in LINE_METRIC_COLUMNS}
assert all(varies.values()), f'constant column being aggregated: {[c for c, v in varies.items() if not v]}'
print(f'All {len(LINE_METRIC_COLUMNS)} metric columns vary within a basket')

# And nothing constant per customer sneaked in.
constant = [c for c in line_metrics.columns
            if c not in ('customer_id',)
            and line_metrics.groupby('customer_id')[c].nunique(dropna=False).max() <= 1]
print(f'Constant-per-customer columns present : {constant}')
print('  (invoice, country, churn and date are inputs carried from the source,')
print('   not aggregated — LINE_METRIC_COLUMNS is what stage 2 consumes)')

In [ ]:
# ---- Stage 2: aggregation ---------------------------------------------------
aggregates = aggregate_line_metrics(line_metrics)
print(f'aggregates : {aggregates.shape[0]:,} rows x {aggregates.shape[1]} columns '
      f'({len(LINE_METRIC_COLUMNS)} metrics x {len(AGGREGATIONS)} aggregations)')
assert list(aggregates.columns) == AGGREGATE_COLUMNS, 'unexpected aggregate columns'
assert len(aggregates) == lines['customer_id'].nunique(), 'not one row per customer'

# Recompute every one of the 50 by hand for a single customer.
CHECK_CUSTOMER = 14793
basket = line_metrics[line_metrics['customer_id'] == CHECK_CUSTOMER]

by_hand = {}
for col in LINE_METRIC_COLUMNS:
    values = basket[col]
    for agg in AGGREGATIONS:
        by_hand[f'{col}_{agg}'] = (values.sum(min_count=1) if agg == 'sum'
                                   else getattr(values, agg)())
by_hand = pd.Series(by_hand)[AGGREGATE_COLUMNS]

mismatch = (by_hand - aggregates.loc[CHECK_CUSTOMER]).abs().max()
assert mismatch < 1e-9, f'aggregation mismatch: {mismatch}'
print(f'\nCustomer {CHECK_CUSTOMER} ({len(basket)} line items): all '
      f'{len(AGGREGATE_COLUMNS)} aggregates match a hand recomputation')

# Shown for two metrics, so the five aggregations are visible as numbers.
shown = pd.DataFrame(
    {agg: [by_hand[f'{c}_{agg}'] for c in ('prior_units', 'price_vs_prior_avg_price')]
     for agg in AGGREGATIONS},
    index=['prior_units', 'price_vs_prior_avg_price'])
display(shown.round(3))

In [ ]:
# ---- Stage 2b: the customers whose whole basket has no history --------------
# Every product in these baskets is making its first-ever appearance, so all four
# ratios are NaN on every line. pandas sums an all-NaN group to 0.0, which would
# hand them a fabricated zero while min/max/mean/median correctly read NaN — and 0
# is a meaningful value for a ratio, so the fabrication would be invisible.
blank = line_metrics.groupby('customer_id')['qty_share_of_prior_units'].apply(
    lambda s: s.isna().all())
blank = blank[blank].index
print(f'Customers with no usable history : {len(blank):,}')

ratio_aggregates = [f'qty_share_of_prior_units_{a}' for a in AGGREGATIONS]
blank_rows = aggregates.loc[blank, ratio_aggregates]
assert blank_rows.isna().all().all(), 'an aggregation invented a value for a blank basket'
print(f'All five aggregations are NaN for every one of them (min_count=1 on the sum)')

# The naive version, for contrast — this is what the sum would have been.
naive = line_metrics.groupby('customer_id')['qty_share_of_prior_units'].sum()
print(f'\nWithout min_count=1, sum would report '
      f'{naive.loc[blank].iloc[0]:.1f} for these customers instead of NaN')

# The prior_* columns behave differently and should: a total with no history is a
# genuine 0 (nothing had sold yet), not an undefined.
print(f'prior_units_sum for the same customers : '
      f'{aggregates.loc[blank, "prior_units_sum"].unique()}  <- 0, not NaN, by design')

In [ ]:
# ---- Stage 3a: average the product vectors per customer ---------------------
embedding_means = aggregate_embeddings(line_metrics, products, product_embeddings)
print(f'embedding_means : {embedding_means.shape[0]:,} rows x '
      f'{embedding_means.shape[1]} columns')
assert list(embedding_means.columns) == EMBEDDING_COLUMNS, 'unexpected embedding columns'
assert np.isfinite(embedding_means.to_numpy()).all(), 'a mean is NaN or Inf'

# By hand for the same customer: index the catalogue matrix to this basket's rows
# and average. Averaging over LINE ITEMS, so a product on two lines counts twice.
positions = pd.Index(products['stock_code']).get_indexer(basket['stock_code'])
by_hand_vector = product_embeddings[positions].mean(axis=0)
from_module = embedding_means.loc[CHECK_CUSTOMER].to_numpy()

print(f'\nCustomer {CHECK_CUSTOMER}: max |by hand - aggregate_embeddings| = '
      f'{np.abs(by_hand_vector - from_module).max():.2e}')
assert np.abs(by_hand_vector - from_module).max() < 1e-6, 'embedding mean mismatch'

# A mean of unit vectors is not unit length, and the shortfall is real signal:
# it falls as a basket gets more varied.
norms = np.linalg.norm(embedding_means.to_numpy(), axis=1)
print(f'Mean-vector norms : min {norms.min():.3f}, median {np.median(norms):.3f}, '
      f'max {norms.max():.3f}  (1.0 only for a single-description basket)')


# ---- Stage 3b: PCA is deliberately NOT applied here -------------------------
# The 384 means go into the dataset as they are. PCA is fitted on the training
# customers only, after the train/test split, so the test set cannot influence
# the components. See reduce_customer_embeddings / reduce_embeddings.
print(f'\nKept {embedding_means.shape[1]} embedding means unreduced; '
      f'PCA is deferred until after the split')


In [ ]:
# ---- Stage 4: one-hot encode, then join the customer-level columns ----------
features_encoded = one_hot_encode(features)

# Each dummy group must decode back to the original column, and exactly the
# baseline customers must have zeros across the group.
for column, baseline in ONE_HOT_BASELINES.items():
    assert features[column].value_counts().idxmax() == baseline, \
        f'{baseline!r} is not the most frequent {column}'
    values = sorted(v for v in features[column].unique() if v != baseline)
    names = [dummy_column_name(column, v) for v in values]
    block = features_encoded[names]
    assert set(np.unique(block.to_numpy())) <= {0, 1}, f'{column} dummies are not 0/1'
    assert (block.sum(axis=1) == (features[column] != baseline)).all(), \
        f'{column}: a baseline customer has a 1, or a non-baseline customer has none'
    for v, name in zip(values, names):
        assert (features_encoded[name] == (features[column] == v)).all(), name
    print(f'{column:<12}: {len(names):>2} dummies, baseline {baseline!r} '
          f'({(features[column] == baseline).mean():.1%} of customers)')

# The plain columns must arrive untouched; the originals must be gone.
plain = [c for c in features.columns
         if c not in ONE_HOT_BASELINES and c not in DROPPED_COLUMNS]
pd.testing.assert_frame_equal(features_encoded[plain], features[plain])
assert not set(ONE_HOT_BASELINES) & set(features_encoded.columns), 'an original survived'
assert not set(DROPPED_COLUMNS) & set(features_encoded.columns), 'a dropped column survived'
print(f'{len(plain)} plain customer-level columns unchanged, '
      f'{features_encoded.shape[1] - len(plain)} dummies added, '
      f'{sorted(set(features.columns) - set(plain))} removed')

dataset_piecewise = (features_encoded
                     .merge(aggregates, on='customer_id', how='left', validate='one_to_one')
                     .merge(embedding_means, on='customer_id', how='left',
                            validate='one_to_one'))

pd.testing.assert_frame_equal(dataset_piecewise[features_encoded.columns], features_encoded)
print(f'All {features_encoded.shape[1]} customer-level columns joined unchanged '
      f'(assert_frame_equal against the encoded table)')

assert len(dataset_piecewise) == len(features), 'the join changed the row count'
assert not dataset_piecewise['customer_id'].duplicated().any(), 'customer_id is not unique'
print(f'Grain preserved : {len(dataset_piecewise):,} rows, one per customer')

# ---- The module end to end --------------------------------------------------
dataset = build_modelling_dataset(lines, model=sbert)
check_modelling_dataset(dataset, lines, model=sbert)
pd.testing.assert_frame_equal(dataset, dataset_piecewise)
print('\nbuild_modelling_dataset reproduces the four stages exactly, '
      'and its own checks pass')

print(f'\nModelling dataset : {dataset.shape[0]:,} rows x {dataset.shape[1]} columns')
print(f'  {features_encoded.shape[1]:>3} customer-level  ({len(plain)} plain + '
      f'{features_encoded.shape[1] - len(plain)} one-hot dummies)')
print(f'  {len(AGGREGATE_COLUMNS):>3} aggregates      ({len(LINE_METRIC_COLUMNS)} metrics x {len(AGGREGATIONS)})')
print(f'  {len(EMBEDDING_COLUMNS):>3} embedding means (unreduced; PCA fitted after the split)')

In [ ]:
# The finished table: one row per customer, a slice of each column group.
SLICE = (['customer_id', 'churn', 'first_date', 'weekday_0', 'time_of_day_morning',
          'country_Germany', 'n_lines', 'total_spend']
         + [f'prior_units_{a}' for a in ('min', 'max', 'mean')]
         + ['qty_share_of_prior_units_sum', 'price_vs_prior_avg_price_median']
         + EMBEDDING_COLUMNS[:3])

print(f'{len(SLICE)} of {dataset.shape[1]} columns shown\n')
with pd.option_context('display.width', 260, 'display.max_colwidth', 22):
    display(dataset[SLICE].head(8).round(3))

In [ ]:
dataset.head()

## The train / test split

`src/split_data.py` splits the modelling table by `first_date` (earliest 80% train, rest test), fits PCA on the training customers' mean embeddings and projects both parts with it. The cells below build the split from the `dataset` above by calling the module, then check each property listed under *The split*. The last check compares the result with the files `python src/split_data.py` wrote, so run that script first.

In [ ]:
import joblib
from split_data import (PCA_FILE, TEST_FILE, TRAIN_FILE, check_split, embedding_matrix,
                        fit_and_apply_pca, time_split)
from description_embeddings import pca_column_names, reduce_embeddings

train, test, cutoff = time_split(dataset)
train_out, test_out, pca = fit_and_apply_pca(train, test)

# The module's own assertions first: a cell that gets past this has already passed them.
check_split(dataset, train, test, train_out, test_out, pca, cutoff)
print(f'check_split passes   : train {train_out.shape}, test {test_out.shape}')

In [ ]:
# ---- 1. Complete and disjoint ------------------------------------------------
ids_train, ids_test = set(train['customer_id']), set(test['customer_id'])
assert not ids_train & ids_test, 'a customer is in both parts'
assert ids_train | ids_test == set(dataset['customer_id']), 'a customer is in neither part'
assert len(train) + len(test) == len(dataset), 'rows were duplicated or lost'
assert not train['customer_id'].duplicated().any() and not test['customer_id'].duplicated().any()
print(f'Partition : {len(train):,} train + {len(test):,} test = {len(dataset):,} customers, '
      f'test share {len(test) / len(dataset):.1%} (requested 20%)')

# ---- 2. Ordered in time -------------------------------------------------------
latest_train = pd.to_datetime(train['first_date']).max()
earliest_test = pd.to_datetime(test['first_date']).min()
assert latest_train < cutoff <= earliest_test, 'train does not end before test begins'
at_cutoff = pd.to_datetime(dataset['first_date']) == cutoff
assert (test['first_date'].isin(dataset.loc[at_cutoff, 'first_date'])).sum() == at_cutoff.sum(), \
    'customers at the cutoff timestamp were split across the parts'
print(f'Time order: latest train {latest_train}  <  cutoff {cutoff}  <=  earliest test {earliest_test}')
print(f'            {at_cutoff.sum()} customer(s) at the cutoff timestamp, all in test')

In [ ]:
# ---- 3. Reproducible ----------------------------------------------------------
train2, test2, cutoff2 = time_split(dataset)
train_out2, test_out2, pca2 = fit_and_apply_pca(train2, test2)
assert cutoff2 == cutoff
pd.testing.assert_frame_equal(train_out2, train_out)
pd.testing.assert_frame_equal(test_out2, test_out)
assert np.array_equal(pca2.components_, pca.components_), 'PCA components differ on a re-run'
print('A second run gives identical train, test and PCA components')

# ---- 4. PCA fitted on train only ---------------------------------------------
all_reduced, pca_all = reduce_embeddings(embedding_matrix(dataset))
assert np.allclose(pca.mean_, embedding_matrix(train).mean(axis=0)), \
    'the PCA centre is not the training mean'
assert not np.allclose(pca.mean_, pca_all.mean_), 'the PCA centre equals the all-customer mean'
assert np.allclose(test_out[pca_column_names(pca.n_components_)].to_numpy(),
                   pca.transform(embedding_matrix(test)), atol=1e-4), \
    'test was not projected with the training PCA'
print(f'PCA fitted on train: {pca.n_components_} components '
      f'({pca.explained_variance_ratio_.sum():.1%} of variance)')
print(f'PCA fitted on all  : {pca_all.n_components_} components (for comparison only; not used)')

In [ ]:
# ---- 5. Nothing else altered --------------------------------------------------
assert list(train_out.columns) == list(test_out.columns), 'column sets differ'
assert train_out.dtypes.equals(test_out.dtypes), 'dtypes differ'
assert not set(EMBEDDING_COLUMNS) & set(train_out.columns), 'raw embeddings survived'

ID_COLUMNS = ['customer_id']
assert not set(ID_COLUMNS) & set(train_out.columns), 'an id column survived in train'
assert not set(ID_COLUMNS) & set(test_out.columns), 'an id column survived in test'
assert 'churn' in train_out.columns and 'churn' in test_out.columns, 'the label is missing'

kept = [c for c in dataset.columns if c not in EMBEDDING_COLUMNS and c not in ID_COLUMNS]
original = dataset.set_index('customer_id')
for name, part, out in (('train', train, train_out), ('test', test, test_out)):
    pd.testing.assert_frame_equal(
        out[kept].reset_index(drop=True),
        original.loc[part['customer_id'], kept].reset_index(drop=True))
n_pca = pca.n_components_
print(f'Columns : {train_out.shape[1]} in both parts = {len(kept)} plain '
      f'(churn + first_date + customer-grain + aggregates) + {n_pca} pca_*')
print(f'customer_id removed (first_date kept for time-ordered CV, not a feature); '
      f'every other non-embedding value equals the modelling table\'s value for the '
      f'same customer, row for row')

# ---- 6. One-hot columns -------------------------------------------------------
dummies = [c for c in train_out.columns if c.startswith(('weekday_', 'time_of_day_', 'country_'))]
assert len(dummies) == 21, len(dummies)
for name, out in (('train', train_out), ('test', test_out)):
    assert set(np.unique(out[dummies].to_numpy())) <= {0, 1}, f'{name}: a dummy is not 0/1'
    for column in ONE_HOT_BASELINES:
        group = [c for c in dummies if c.startswith(f'{column}_')]
        assert (out[group].sum(axis=1) <= 1).all(), f'{name}: {column} has two categories'
counts = pd.DataFrame({'train': train_out[dummies].sum(), 'test': test_out[dummies].sum()})
print(f'{len(dummies)} dummies, all 0/1, at most one per group')
print('Dummies with no customer in one part:')
display(counts[(counts == 0).any(axis=1)])

In [ ]:
# ---- 7. Base rates and missingness (reported, not asserted) -------------------
no_history = 'qty_share_of_prior_units_sum'
report = pd.DataFrame({
    'customers': [len(train_out), len(test_out)],
    'first_date from': [train['first_date'].min(), test['first_date'].min()],
    'first_date to': [train['first_date'].max(), test['first_date'].max()],
    'churn rate': [train_out['churn'].mean(), test_out['churn'].mean()],
    'no product history': [train_out[no_history].isna().mean(), test_out[no_history].isna().mean()],
}, index=['train', 'test'])
with pd.option_context('display.width', 200):
    display(report)
print(f'Churn-rate gap, test minus train : {test_out["churn"].mean() - train_out["churn"].mean():+.2%}')

# ---- 8. The written files match what the code produces ----------------------------
for path in (TRAIN_FILE, TEST_FILE, PCA_FILE):
    assert path.exists(), f'{path.name} is missing — run python src/split_data.py first'
for name, out, path in (('train', train_out, TRAIN_FILE), ('test', test_out, TEST_FILE)):
    on_disk = pd.read_csv(path, parse_dates=['first_date'])
    pd.testing.assert_frame_equal(on_disk, out, check_dtype=False, check_exact=False,
                                  rtol=1e-5, atol=1e-5)
    print(f'{path.name:<10}: {on_disk.shape} matches the in-memory {name} set')
saved = joblib.load(PCA_FILE)
assert np.allclose(saved.components_, pca.components_, atol=1e-8), 'pca.joblib differs'
print(f'{PCA_FILE.name:<10}: {saved.n_components_} components match')

In [ ]:
train_out.head()